# 🐍 Nivelación Python · Sesión 2 — Orientación a Objetos y el Misterio de `self`
### Módulo de Nivelación Python — Sesión 2 · material transversal (S00)

> **Audiencia:** Estudiantes con POO sólida en Java que asumen que `this` es implícito y mágico.  
> **Duración estimada:** 35–40 minutos de live coding.  
> **Objetivo:** Mapear los conceptos de Java a Python, desmitificar `self` con el modelo de despacho de métodos, y construir una clase completa con estado interno mutable.

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Comprender** qué es `self` y por qué Python lo hace explícito donde Java usa un `this` implícito.
2. **Identificar** la correspondencia entre los conceptos de POO de Java y su equivalente en Python.
3. **Implementar** una clase completa con `__init__`, atributos de instancia y métodos propios.
4. **Analizar** el contrato de los métodos especiales `__dunder__` y cuándo conviene implementarlos.
5. **Resolver** el diseño de una estructura de datos propia respetando la interfaz que Python espera.

---
## Parte 1 — Mapa de Java a Python: Lo que ya sabes, renombrado

La buena noticia: los conceptos de POO que aprendiste en Java **están todos en Python**. Solo tienen nombres o sintaxis diferentes.

| Concepto | Java | Python |
|---|---|---|
| Definir clase | `class Punto { }` | `class Punto:` |
| Constructor | `public Punto(int x, int y) {}` | `def __init__(self, x, y):` |
| Atributo de instancia | `this.x = x;` | `self.x = x` |
| Método de instancia | `public int getX() { return this.x; }` | `def get_x(self): return self.x` |
| Crear objeto | `Punto p = new Punto(3, 4);` | `p = Punto(3, 4)` |
| Llamar método | `p.getX()` | `p.get_x()` |
| Herencia | `class B extends A {}` | `class B(A):` |
| Método toString | `@Override public String toString()` | `def __str__(self):` |
| Método equals | `@Override public boolean equals(Object o)` | `def __eq__(self, other):` |

La diferencia más llamativa: en Python **no hay modificadores de acceso** (`public`, `private`, `protected`).  
La convención es usar un guión bajo prefijo para indicar "uso interno": `_atributo_privado`.

---
## Parte 2 — El Misterio de `self`: La Explicación Técnica

### El modelo Java (lo que asumes)

En Java, cuando escribes dentro de un método `this.x = 5`, el compilador **inyecta automáticamente** la referencia al objeto actual. `this` nunca aparece en la firma del método:

```java
public class Punto {
    int x;
    //           ↓ el compilador agrega 'this' de forma invisible
    public void setX(int x) {
        this.x = x;  // this llegó solo, no lo declaraste
    }
}
```

### ¿Por qué Python exige `self` explícito?

Python es un lenguaje donde **todo es explícito**. Cuando llamas a un método de instancia:

```python
p.set_x(5)
```

Python lo traduce internamente a una llamada a función regular:

```python
Punto.set_x(p, 5)   # ← Python pasa el objeto como PRIMER argumento
```

Por tanto, el método **tiene que declarar** ese primer parámetro para recibirlo. Por convención universal se llama `self`, pero técnicamente podría llamarse de cualquier forma (aunque nunca deberías usar otro nombre).

```
p.set_x(5)
    │       │
    │       └─── argumento explícito del programador
    └─────────── Python lo convierte en: Punto.set_x(p, 5)
                                                     ↑
                                              self recibe esto
```

> **Resumen:** `self` no es magia. Es el objeto que llama al método, pasado como primer argumento. Python te obliga a declararlo para que sea **visible y explícito**.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: *"Si `self` se pasa solo, ¿por qué hay que escribirlo en la definición del método pero no en la llamada?"* — Pídeles que llamen al método como `Clase.metodo(objeto)` y verán que es la misma cosa.

In [ ]:
# === Demostración: método de instancia vs. llamada directa a la clase ===
# Ambas formas son EQUIVALENTES. Python hace la traducción automáticamente.

class Punto:
    """Representa un punto en el plano cartesiano."""

    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

    def desplazar(self, dx: float, dy: float) -> None:
        """Desplaza el punto sumando dx a x y dy a y."""
        self.x += dx
        self.y += dy

    def __str__(self) -> str:
        return f"Punto({self.x}, {self.y})"


p = Punto(3.0, 4.0)
print("Objeto creado:", p)

# Forma normal: método de instancia
p.desplazar(1.0, 2.0)
print("Después de p.desplazar(1, 2):", p)

# Forma interna: llamada directa a la clase (exactamente lo mismo)
Punto.desplazar(p, 10.0, 20.0)
print("Después de Punto.desplazar(p, 10, 20):", p)

# ↑ Esto prueba que p.desplazar(dx, dy) == Punto.desplazar(p, dx, dy)

---
## Parte 3 — `__init__`: El Constructor de Python

### ¿Qué hace realmente `__init__`?

En Java, el constructor tiene el mismo nombre que la clase y *crea* el objeto:  
```java
public Punto(int x, int y) { ... }  // crea Y configura
```

En Python, la creación del objeto la maneja `__new__` (que raramente tocamos). `__init__` solo **inicializa** el objeto ya creado. Por eso recibe `self` — el objeto ya existe cuando `__init__` se ejecuta.

```
Punto(3, 4)
  │
  ├── 1) Python llama a Punto.__new__(Punto)   → crea el objeto vacío
  └── 2) Python llama a Punto.__init__(obj, 3, 4) → configura los atributos
```

### Los atributos de instancia NO se declaran fuera del constructor

En Java declaras los campos en el cuerpo de la clase:
```java
class Punto {
    int x;   // campo declarado aquí
    int y;
    public Punto(int x, int y) { this.x = x; }
}
```

En Python, los atributos de instancia se crean directamente dentro de `__init__` con `self.nombre = valor`. No existe una sección separada de "declaración de campos".

In [ ]:
# === Comparativa de constructores con y sin valores por defecto ===

class Rectangulo:
    """
    Representa un rectángulo con ancho y alto.
    Los atributos de instancia se crean en __init__ con self.
    """

    # Python equivale a: public Rectangulo(double ancho, double alto)
    # Los valores por defecto son equivalentes al overloading en Java
    def __init__(self, ancho: float = 1.0, alto: float = 1.0) -> None:
        self.ancho = ancho   # equivale a this.ancho = ancho en Java
        self.alto = alto

    def area(self) -> float:
        """Retorna el área del rectángulo."""
        return self.ancho * self.alto

    def perimetro(self) -> float:
        """Retorna el perímetro del rectángulo."""
        return 2 * (self.ancho + self.alto)

    def escalar(self, factor: float) -> None:
        """Escala el rectángulo multiplicando ancho y alto por factor."""
        self.ancho *= factor   # self modifica el ESTADO del objeto
        self.alto *= factor

    def __str__(self) -> str:
        return f"Rectangulo(ancho={self.ancho}, alto={self.alto})"


# Java: Rectangulo r1 = new Rectangulo();  (constructor sin args)
r1 = Rectangulo()
print(f"r1 = {r1}  |  área = {r1.area()}")

# Java: Rectangulo r2 = new Rectangulo(5.0, 3.0);
r2 = Rectangulo(5.0, 3.0)
print(f"r2 = {r2}  |  área = {r2.area()}  |  perímetro = {r2.perimetro()}")

# Los dos objetos son independientes (estado interno separado)
r2.escalar(2.0)
print(f"\nr2 después de escalar(2.0): {r2}")
print(f"r1 no se modificó: {r1}")   # r1 permanece intacto

---
## Parte 4 — Proyecto: La clase `MatrizPersonalizada`

Vamos a construir una clase completa que encapsula una matriz 2D. Este es el tipo de TDA (Tipo de Dato Abstracto) que encontrarás en el curso de Estructuras de Datos.

La clase tendrá:
- Un constructor que recibe dimensiones y un valor inicial
- Métodos para leer y escribir celdas individuales
- Un método para imprimir la matriz con formato
- Un método para recorrer toda la matriz
- Validación de índices para detectar errores temprano

### Anatomía que construiremos

```
MatrizPersonalizada
├── __init__(filas, cols, valor_inicial)   ← constructor
├── _validar_indices(fila, col)             ← método privado de apoyo
├── obtener(fila, col)                      ← leer celda
├── insertar(fila, col, valor)              ← escribir celda  
├── recorrer()                              ← iterar toda la matriz
├── imprimir()                              ← visualizar con formato
└── __str__()                               ← representación textual
```

In [ ]:
# === Implementación de MatrizPersonalizada ===
# Construimos la clase paso a paso, método por método.

from typing import Any, Generator


class MatrizPersonalizada:
    """
    Matriz 2D de tamaño fijo con acceso por índices (fila, col).
    Encapsula una lista de listas Python, exponiendo una API segura.
    """

    # ------------------------------------------------------------------
    # CONSTRUCTOR
    # En Java: public MatrizPersonalizada(int filas, int cols, Object valorInicial)
    # ------------------------------------------------------------------
    def __init__(self, filas: int, cols: int, valor_inicial: Any = 0) -> None:
        """
        Inicializa la matriz con dimensiones filas × cols.

        :param filas: número de filas (debe ser > 0)
        :param cols: número de columnas (debe ser > 0)
        :param valor_inicial: valor con que se rellena cada celda (default 0)
        """
        if filas <= 0 or cols <= 0:
            raise ValueError(f"Dimensiones inválidas: {filas}×{cols}. Deben ser > 0.")

        self.filas: int = filas                   # self almacena el estado
        self.cols: int = cols                     # visible en todos los métodos
        # Comprensión anidada para crear filas independientes (¡Notebook 1!)
        self._datos: list[list[Any]] = [
            [valor_inicial] * cols for _ in range(filas)
        ]

    # ------------------------------------------------------------------
    # MÉTODO PRIVADO DE APOYO
    # Prefijo _ indica "no usar desde fuera de la clase" (convención Python)
    # En Java sería: private void validarIndices(int fila, int col)
    # ------------------------------------------------------------------
    def _validar_indices(self, fila: int, col: int) -> None:
        """Lanza IndexError si (fila, col) está fuera de rango."""
        if not (0 <= fila < self.filas):
            raise IndexError(
                f"Fila {fila} fuera de rango. Válido: 0..{self.filas - 1}"
            )
        if not (0 <= col < self.cols):
            raise IndexError(
                f"Columna {col} fuera de rango. Válido: 0..{self.cols - 1}"
            )

    # ------------------------------------------------------------------
    # LEER UNA CELDA
    # En Java: public Object obtener(int fila, int col)
    # ------------------------------------------------------------------
    def obtener(self, fila: int, col: int) -> Any:
        """Retorna el valor en la celda (fila, col)."""
        self._validar_indices(fila, col)    # self llama a su propio método
        return self._datos[fila][col]       # self accede a su propio estado

    # ------------------------------------------------------------------
    # ESCRIBIR UNA CELDA
    # En Java: public void insertar(int fila, int col, Object valor)
    # ------------------------------------------------------------------
    def insertar(self, fila: int, col: int, valor: Any) -> None:
        """Asigna 'valor' en la celda (fila, col)."""
        self._validar_indices(fila, col)
        self._datos[fila][col] = valor      # modifica el estado interno

    # ------------------------------------------------------------------
    # RECORRER TODA LA MATRIZ (generador)
    # Retorna tuplas (fila, col, valor) para cada celda
    # ------------------------------------------------------------------
    def recorrer(self) -> Generator[tuple[int, int, Any], None, None]:
        """Genera tuplas (fila, col, valor) para cada celda de la matriz."""
        for i in range(self.filas):
            for j in range(self.cols):
                yield i, j, self._datos[i][j]  # 'yield' pausa y entrega un valor

    # ------------------------------------------------------------------
    # IMPRIMIR CON FORMATO
    # ------------------------------------------------------------------
    def imprimir(self) -> None:
        """Imprime la matriz con índices de fila y columna."""
        # Encabezado de columnas
        print("      " + "  ".join(f"C{j:02d}" for j in range(self.cols)))
        print("      " + "---" * self.cols)
        for i, fila in enumerate(self._datos):
            fila_str = "  ".join(f"{v!s:>4}" for v in fila)
            print(f"F{i:02d} | {fila_str}")

    # ------------------------------------------------------------------
    # __str__: lo que print(objeto) muestra
    # Equivale a @Override public String toString() en Java
    # ------------------------------------------------------------------
    def __str__(self) -> str:
        return f"MatrizPersonalizada({self.filas}×{self.cols})"

### Uso de la clase: el cliente de la API

El código que *usa* una clase no debe acceder a `_datos` directamente — solo usa los métodos públicos. Esto es el mismo principio de encapsulamiento que aprendiste en Java.

In [ ]:
# === Creación y uso básico ===

# Java: MatrizPersonalizada m = new MatrizPersonalizada(3, 4, 0);
m = MatrizPersonalizada(3, 4, 0)
print(m)           # usa __str__
print(f"Dimensiones: {m.filas} filas × {m.cols} columnas")

print("\nMatriz recién creada:")
m.imprimir()

In [ ]:
# === Insertar valores y verificar que el estado interno cambia ===

# Llenamos la diagonal principal con 1s (como una identidad parcial)
for k in range(min(m.filas, m.cols)):
    m.insertar(k, k, 1)    # self._datos[k][k] = 1  ← modifica estado interno

print("Después de insertar 1s en la diagonal:")
m.imprimir()

# Insertar valores específicos
m.insertar(0, 3, 99)
m.insertar(2, 1, 42)

print("\nDespués de inserciones adicionales:")
m.imprimir()

In [ ]:
# === Recorrer la matriz con el generador ===
# Solo imprimimos las celdas con valor distinto de 0

print("Celdas con valor ≠ 0:")
for fila, col, valor in m.recorrer():
    if valor != 0:
        print(f"  m[{fila}][{col}] = {valor}")

# Leer una celda específica
v = m.obtener(0, 3)
print(f"\nValor en (0, 3): {v}")

In [ ]:
# === Demostración: los objetos tienen estado INDEPENDIENTE ===
# Equivale a crear dos objetos distintos en Java y verificar que no comparten memoria

m1 = MatrizPersonalizada(2, 2, 0)
m2 = MatrizPersonalizada(2, 2, 0)

m1.insertar(0, 0, 100)   # solo m1 cambia

print("m1 después de insertar 100 en (0,0):")
m1.imprimir()

print("\nm2 no se modificó (estado independiente):")
m2.imprimir()

# Confirmamos que self apunta a objetos distintos
print(f"\nid(m1) = {id(m1)}")
print(f"id(m2) = {id(m2)}")
print(f"¿Son el mismo objeto? {m1 is m2}")

In [ ]:
# === Demostración: qué pasa si NO usamos self correctamente ===
# Este ejemplo muestra el error que ocurre si olvidamos 'self'

class MatrizRota:
    """Clase con un bug intencional: atributo local en lugar de self."""

    def __init__(self, filas: int, cols: int) -> None:
        # BUG: 'datos' es una variable LOCAL de __init__, no un atributo
        # No sobrevive fuera de este método
        datos = [[0] * cols for _ in range(filas)]  # ← NO self.datos
        self.filas = filas
        self.cols = cols

    def obtener(self, fila: int, col: int) -> int:
        return self.datos[fila][col]  # AttributeError: no existe self.datos


try:
    mr = MatrizRota(3, 3)
    print(mr.obtener(0, 0))  # lanzará AttributeError
except AttributeError as e:
    print(f"Error capturado: {e}")
    print()
    print("La causa: 'datos' se creó como variable local en __init__.")
    print("Sin 'self.', el valor se pierde cuando __init__ termina.")
    print("La corrección: usar self.datos = ... en __init__.")

---
## Parte 5 — Métodos Especiales: El Contrato `__dunder__`

Python tiene una convención llamada **dunder methods** (de *double underscore*: `__`). Son métodos especiales que Python llama automáticamente ante ciertas operaciones. Son el equivalente a las interfaces `Comparable`, `Iterable` y `toString()` de Java.

| Operación | Python llama a... | Equivalente Java |
|---|---|---|
| `print(obj)` | `__str__(self)` | `toString()` |
| `repr(obj)` | `__repr__(self)` | — (debug string) |
| `a == b` | `__eq__(self, other)` | `equals(Object o)` |
| `len(obj)` | `__len__(self)` | `size()` |
| `obj[i]` | `__getitem__(self, i)` | `get(i)` |
| `obj[i] = v` | `__setitem__(self, i, v)` | `set(i, v)` |
| `for x in obj` | `__iter__(self)` | `Iterator<T> iterator()` |

In [ ]:
# === Extender MatrizPersonalizada con métodos dunder ===
# Añadimos __len__, __getitem__ y __eq__ para que la clase sea más idiomática.

class MatrizPersonalizada:
    """Matriz 2D de tamaño fijo con interfaz idiomática de Python."""

    def __init__(self, filas: int, cols: int, valor_inicial: Any = 0) -> None:
        if filas <= 0 or cols <= 0:
            raise ValueError(f"Dimensiones inválidas: {filas}×{cols}.")
        self.filas = filas
        self.cols = cols
        self._datos: list[list[Any]] = [
            [valor_inicial] * cols for _ in range(filas)
        ]

    def _validar_indices(self, fila: int, col: int) -> None:
        if not (0 <= fila < self.filas):
            raise IndexError(f"Fila {fila} fuera de rango.")
        if not (0 <= col < self.cols):
            raise IndexError(f"Columna {col} fuera de rango.")

    def obtener(self, fila: int, col: int) -> Any:
        self._validar_indices(fila, col)
        return self._datos[fila][col]

    def insertar(self, fila: int, col: int, valor: Any) -> None:
        self._validar_indices(fila, col)
        self._datos[fila][col] = valor

    def recorrer(self) -> Generator[tuple[int, int, Any], None, None]:
        for i in range(self.filas):
            for j in range(self.cols):
                yield i, j, self._datos[i][j]

    def imprimir(self) -> None:
        print("      " + "  ".join(f"C{j:02d}" for j in range(self.cols)))
        print("      " + "---" * self.cols)
        for i, fila in enumerate(self._datos):
            fila_str = "  ".join(f"{v!s:>4}" for v in fila)
            print(f"F{i:02d} | {fila_str}")

    # --- Métodos dunder nuevos ---

    def __len__(self) -> int:
        """Retorna el total de celdas. Permite: len(matriz)."""
        return self.filas * self.cols

    def __getitem__(self, indices: tuple[int, int]) -> Any:
        """Permite leer con la sintaxis: matriz[fila, col]."""
        fila, col = indices
        return self.obtener(fila, col)

    def __setitem__(self, indices: tuple[int, int], valor: Any) -> None:
        """Permite escribir con la sintaxis: matriz[fila, col] = valor."""
        fila, col = indices
        self.insertar(fila, col, valor)

    def __eq__(self, other: object) -> bool:
        """Dos matrices son iguales si tienen las mismas dimensiones y datos."""
        if not isinstance(other, MatrizPersonalizada):
            return NotImplemented
        return self._datos == other._datos

    def __str__(self) -> str:
        return f"MatrizPersonalizada({self.filas}×{self.cols})"

    def __repr__(self) -> str:
        """Representación técnica para el debugger."""
        return f"MatrizPersonalizada(filas={self.filas}, cols={self.cols})"


# --- Demostración de los nuevos dunder ---
m = MatrizPersonalizada(3, 4, 0)

# __len__
print(f"len(m) = {len(m)} celdas")   # 3 × 4 = 12

# __setitem__ y __getitem__
m[1, 2] = 77          # Python llama: m.__setitem__((1, 2), 77)
print(f"m[1, 2] = {m[1, 2]}")  # Python llama: m.__getitem__((1, 2))

# __eq__
m2 = MatrizPersonalizada(3, 4, 0)
m2[1, 2] = 77
print(f"\n¿m == m2? {m == m2}")   # True: mismos datos

m2[0, 0] = 999
print(f"¿m == m2 después de cambiar m2[0,0]? {m == m2}")  # False

# __repr__
print(f"\nrepr(m) = {repr(m)}")

---
## Resumen Visual

| Concepto | Java | Python |
|---|---|---|
| `this` | Inyectado implícitamente por el compilador | `self` declarado explícitamente |
| `p.metodo(x)` internamente | `Clase.metodo(this, x)` | `Clase.metodo(p, x)` — idéntico |
| Constructor | `public Clase(args) {}` | `def __init__(self, args):` |
| Atributo de instancia | `int x;` + `this.x = ...` | Solo `self.x = ...` en `__init__` |
| Atributo "privado" | `private int x;` | `self._x` (convención `_`) |
| `toString()` | `@Override public String toString()` | `def __str__(self):` |
| `equals()` | `@Override public boolean equals(Object o)` | `def __eq__(self, other):` |
| Colección iterable | `Iterable<T>` + `iterator()` | `def __iter__(self):` |

---

## Ejercicios de Práctica

In [ ]:
# EJERCICIO 1
# Completa la clase Fraccion. Debe representar una fracción numerador/denominador.
# Implementa:
#   - __init__(self, numerador, denominador)  →  valida que denominador != 0
#   - __str__(self)                           →  retorna "3/4"
#   - sumar(self, otra)                       →  retorna una nueva Fraccion (no modifica self)
#
# Pista para suma: a/b + c/d = (a*d + c*b) / (b*d)
# Resultado esperado:
#   f1 = 1/2
#   f2 = 1/3
#   f1 + f2 = 5/6

class Fraccion:
    def __init__(self, numerador: int, denominador: int) -> None:
        pass  # TU CÓDIGO AQUÍ

    def __str__(self) -> str:
        pass  # TU CÓDIGO AQUÍ

    def sumar(self, otra: "Fraccion") -> "Fraccion":
        pass  # TU CÓDIGO AQUÍ


f1 = Fraccion(1, 2)
f2 = Fraccion(1, 3)
print(f"f1 = {f1}")
print(f"f2 = {f2}")
print(f"f1 + f2 = {f1.sumar(f2)}")

In [ ]:
# EJERCICIO 2
# Dado el siguiente código que usa MatrizPersonalizada,
# ¿cuál es el estado final de la matriz?
# Razona ANTES de ejecutarlo. Luego ejecútalo para verificar.

mx = MatrizPersonalizada(3, 3, 1)

for i in range(mx.filas):
    mx.insertar(i, i, i * 10)   # ¿qué celdas se modifican?

for fila, col, valor in mx.recorrer():
    if fila == col:
        mx.insertar(fila, col, valor + 5)  # ¿y ahora?

print("Estado final:")
mx.imprimir()

# Escribe aquí tu predicción antes de ejecutar:
# F00 = ?, F11 = ?, F22 = ?, resto = ?

In [ ]:
# EJERCICIO 3 — Reflexión sobre self
# ¿Qué error produce este código? ¿Por qué?
# Corrígelo añadiendo self donde corresponda.

class Contador:
    def __init__(self) -> None:
        self.valor = 0

    def incrementar(n: int) -> None:   # ← falta algo en la firma
        self.valor += n                # ← y aquí también

    def obtener_valor() -> int:        # ← ídem
        return self.valor


try:
    c = Contador()
    c.incrementar(5)
    print(c.obtener_valor())
except TypeError as e:
    print(f"Error: {e}")
    print("Explica por qué ocurre y cómo se corrige:")
    # TU RESPUESTA AQUÍ

---
*Módulo de Nivelación Python — Notebook 2 de 4*  
*Siguiente: **Notebook 3 — Ingeniería Básica: Pruebas Unitarias en Jupyter***

## 📚 Lecturas Recomendadas y Práctica

### Textbooks

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Ramalho (Fluent) — *Fluent Python* | 2ª ed. | Cap. 11 | Un objeto pythónico: cómo se diseña una clase |
| Ramalho (Fluent) — *Fluent Python* | 2ª ed. | Cap. 1 | El modelo de datos de Python y los métodos especiales |
| Goodrich, Tamassia & Goldwasser (GTG) — *Data Structures and Algorithms in Python* | 1ª ed. | Cap. 2 | Programación orientada a objetos |

### Recursos gratuitos en línea

- 📄 [Python Tutorial oficial — Clases](https://docs.python.org/es/3/tutorial/classes.html) — incluye la explicación de `self`.
- 📄 [Python Data Model](https://docs.python.org/3/reference/datamodel.html) — el contrato completo de los métodos `__dunder__`.

### Práctica en Codeforces (soporta Python 3)

> 🔍 **Cómo filtrar:** ve a [codeforces.com/problemset](https://codeforces.com/problemset),
> escribe la etiqueta en **Tags** y ajusta **Rating**.

**Escala de dificultad orientativa para este curso:**

| Rating | Nivel | Descripción |
|--------|-------|-------------|
| 800 | ⭐ | Aplicación directa — la mayoría puede resolverlo |
| 1000–1200 | ⭐⭐ | Requiere una pequeña adaptación |
| 1300+ | ⭐⭐⭐ | Combina la idea con otra — desafío |

**Problemas recomendados para este tópico:**

| # | Problema | Rating | Por qué es útil |
|---|----------|--------|-----------------|
| 1 | [4A — Watermelon](https://codeforces.com/problemset/problem/4/A) | ⭐ 800 | Envolver una decisión simple en una función bien nombrada |
| 2 | [467A — George and Accommodation](https://codeforces.com/problemset/problem/467/A) | ⭐ 800 | Modelar entidades con atributos y filtrarlas |
| 3 | [231A — Team](https://codeforces.com/problemset/problem/231/A) | ⭐⭐ 800 | Agregar sobre una colección de objetos |

⚠️ Los dos primeros son el **mínimo esperado**. Los demás son desafío opcional.